# Introduction:

Customer Churn: What Is It?
 When consumers or subscribers stop using a company or service, it's known as customer churn.


 In the telecom sector, consumers can actively switch between a number of service providers.  In this fiercely competitive sector, the telecom industry has an annual churn rate of 15 – 25%.

Because most businesses have a huge number of clients and cannot afford to spend much time with each one, individualized customer retention is difficult.  The greater revenue would be outweighed by the excessive costs.  However, a company could concentrate its customer retention efforts solely on these "high risk" consumers if it could predict which customers are likely to depart in advance.  The ultimate objective is to increase its coverage area and win back more loyal clients.  The client is the key to success in this business.

 Because retaining current customers is far less expensive than acquiring new ones, customer churn is an important statistic.

 Telecom businesses must identify which customers are most likely to leave in order to lower customer attrition.

 One must first create a comprehensive picture of the customers and their interactions across a variety of channels, such as store/branch visits, product purchase histories, customer service calls, Web-based transactions, and social media interactions, to identify early indicators of possible churn.


 Therefore, by managing churn, these companies may be able to develop and prosper in addition to maintaining their market position.  The smaller the beginning cost and the higher the profit, the more consumers they have in their network.  Therefore, lowering client attrition and putting in place an efficient retention strategy are the company's primary goals for success.

 Objective:

 I'll examine the information and attempt to respond to some queries such as:


- What is the percentage of consumers who stay with the active services and those who churn?
- Are there any gender-based trends in Churn Customers?
- Are there any trends or preferences among churn customers according to the kind of services offered?
- Which service categories are the most lucrative?
- What are the most lucrative features and services?
- Numerous additional queries that will come up during the analysis
---

# Here's how we do it:

##**Section 1. Import Libraries, Data and Diagnosis:**

**Step 1: Import libraries and data:**

In [ ]:
import pandas as pd
import numpy as np
import io
import os
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
!pip install catboost
from catboost import CatBoostClassifier


Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.0 MB/s eta 0:00:00


In [ ]:
try:
    df = pd.read_csv("kaggle kernels pull bhartiprasad17/customer-churn-prediction")

    print("Step 1 Output:")
    print("Successfully loaded file into DataFrame 'df'.")
    print(f"DataFrame has {df.shape[0]} rows and {df.shape[1]} columns.")

except Exception as e:
    print(f"Step 1 ERROR: {e}")

Step 1 ERROR: [Errno 2] No such file or directory: 'kaggle kernels pull bhartiprasad17/customer-churn-prediction'


Now that we have the **DataFrame** in memory, let's **look at the first 5 lines** in Step 2 to see if the data looks correct.

**Step 2: Understand the data:**

In [ ]:
print("Step 2 Output:")
print(df.head().to_markdown(index=False, numalign="left", stralign="left"))

Step 2 Output:


NameError: name 'df' is not defined

**Explaination:**
- **df.head()**: This is an extremely common "smoke test." It helps answer the questions:

1. **Is the column name incorrect?** (Example: customerID instead of customer ID). -> Result: The column names seem standard.

2. **Is the data skewed?** -> Result: No, gender contains Female/Male, tenure contains numbers.

3. **Does the data contain any special characters?** -> Result: In the first five rows, it looks clean.

- **Meaning**: The data has been read into the correct structure, and the columns and rows match.

**Step 3: Check the structure of the data that pandas has automatically inferred:**

In [ ]:
print("Step 3 Output:")
buf = io.StringIO()
df.info(buf=buf)
info_str = buf.getvalue()

print(info_str)

**Explaination:**

- **df.info()**: Provides 3 important pieces of information about each column:

- **Column name** (e.g., MonthlyCharges).

- **Non-Null Count**: All 21 columns show 7043 non-null values. This means there is no standard NaN (Not a Number) value.

- **Dtype** (Data Type): This is the most important finding.

- **MonthlyCharges** is float64 -> Correct.

- **TotalCharges** is still referred as object  -> Incorrect!

Object is how pandas refers to a column containing strings (text). The fact that TotalCharges is treated as text while MonthlyCharges is a number is a clear "red flag". This means there is at least one value in the TotalCharges column that is not a number.

We saw **df.info()** say that there are no "non-null" values. This 4th step will clearly reconfirm that.

In [ ]:
print("Step 4 Output:")
print(df.isnull().sum().to_markdown(numalign="left", stralign="left"))

Technical explanation of Step 4

- **df.isnull().sum()**: This function only finds NaN (Not a Number) or None (Python's null type) values.

- **Conflict**: This result states that TotalCharges has $0$ null values. But Step 3 **(df.info())** says **TotalCharges is an object (text)**.

- **Inference**: These two steps combined tell us that the issue with the TotalCharges column is not standard NaN values. The problem is that it contains something that is text but not a number, such as an empty string (' ') or a letter ('abc'). **pandas considers ' ' to be a valid text value, not null**.

This is the step where we resolve the conflict from Steps 3 and 4. We will intentionally coerce the TotalCharges column to a numeric type in this 5th step and see what happens.

In [ ]:
print("Step 5 Output:")
total_charges_numeric = pd.to_numeric(df['TotalCharges'], errors='coerce')
invalid_count = total_charges_numeric.isnull().sum()
print(f"DETECTED: Column 'TotalCharges' has {invalid_count} rows that are not numeric (e.g., empty string ' ').")

print("\nExamples of rows with invalid 'TotalCharges' (usually tenure=0):")

invalid_rows = df[total_charges_numeric.isnull()]
print(invalid_rows.head(5).to_markdown(index=False, numalign="left", stralign="left"))

Technical Explanation of Step 5

- **pd.to_numeric(..., errors='coerce')**: This is the ultimate diagnostic tool. We ask pandas to convert the TotalCharges column to a number.

- **invalid_count = . . . . isnull(). sum()**: Let's count how many values are converted to NaN during type conversion. The result is 11.

- **Discovery**: These are the 11 "troublesome" values. They are empty strings (' ') that df.info() and df.isnull() do not detect.

- **df[total_charges_numeric.isnull()]**: By filtering this way, we can see the 11 original rows that caused the error. We immediately see a pattern: they all have tenure = 0. This makes sense: new customers (0 months) should not yet have a "Total Charge." The system outputted ' ' instead of '0'.

- **Action**: Now that we know the exact problem and how to fix it: we need to turn these 11 values into '0'.


##**Looking Back #1:**

Section 1 provides us a comprehensive overview of our progress after the initial data loading and diagnosis. Here's a summary of its key points:

- **Data Overview:** It confirms that we have successfully loaded the *WA_Fn-UseC_-Telco-Customer-Churn.csv* dataset, which contains 7043 rows and 21 columns related to telecommunications customer information. It also highlights the resolution of a data quality issue where the TotalCharges column, initially an object type with blank entries, was corrected to a numeric type, with blank values set to 0. The customerID column has been identified for removal.

- **Features:** It categorizes the features into demographic information (gender, SeniorCitizen, Partner, Dependents), service information (PhoneService, MultipleLines, InternetService, OnlineSecurity, OnlineBackup, DeviceProtection, TechSupport, StreamingTV, StreamingMovies), and contract & billing information (Contract, PaperlessBilling, PaymentMethod, MonthlyCharges, TotalCharges, tenure).

- **Label:** The target variable, Churn, is identified. It indicates whether a customer has left the company (Yes) or stayed (No). For modeling, these values will be converted to 1 and 0 respectively.

- **Problem Type:** The problem is defined as a Binary Classification task, where the goal is to predict customer churn.

- **Next Steps:** The section outlines that the next phase involves Section 2: Data Visualization, where we will explore data distributions and relationships to gain deeper insights into churn patterns, which will then inform further preprocessing and model development.

##**Section 2. Data Visualization:**

**Step 1: Set up & Prepare Data:**

Before we draw, we need to clean our data.

In [ ]:
df.replace('No internet service', 'No', inplace=True)
df.replace('No phone service', 'No', inplace=True)
df.describe(include='all').T
print("Data is ready!")
print(df.describe(include='all').T)

**Explaination:**

- **Why replace it?** Labels like "No internet service" are too long; when plotted on a graph, they will be cluttered or overlap the text. Grouping into "No" makes the chart cleaner.

**Step 2: Analyze Churn Rate (Target Variable)**

We need to know the "big picture": **How many customers are leaving versus staying?**

In [ ]:
plt.figure(figsize=(6, 6))
churn_counts = df['Churn'].value_counts()


plt.pie(churn_counts, labels=churn_counts.index, autopct='%1.1f%%',
        startangle=90, colors=["#C9E4DE", "#FAEDCD"], explode=(0, 0))

plt.title('Customer Churn Rate')
plt.show()

**Chart analysis:**

- Results: You will see approximately 26.5% of customers leave (Yes) and 73.5% stay (No).

- Insight: This is the baseline rate. When analyzing subgroups (e.g., the fiber optic group), if the churn rate is > 26.5%, that group is experiencing problems.

**Explanation:**

- **explode=(0, 0)**: By setting the values in explode to 0, the two parts of the chart will not separate, resulting in a perfectly round and easy-to-read pie chart.

- **autopct='% 1.1f%%**: Display the percentage on the chart with one decimal place.

**Step 3: Analyze Personal Information (Demographics):**

The question is: **Does gender or age affect the decision to leave?**

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.countplot(data=df, x='gender', hue='Churn', ax=axes[0], palette="pastel")
axes[0].set_title('By gender')

sns.countplot(data=df, x='SeniorCitizen', hue='Churn', ax=axes[1], palette="pastel")
axes[1].set_title('By SeniorCitizen')

sns.countplot(data=df, x='Partner', hue='Churn', ax=axes[2], palette="pastel")
axes[2].set_title('By Partner')

plt.tight_layout()
plt.show()

**Chart analysis:**

- **Gender**: The two columns for Male/Female are equal in height, and the proportion of Orange (Left) is also equivalent. -> Conclusion: Gender doesn't matter.

- **Senior Citizen**: Group 1 (Yes), although smaller in number, has a very high percentage of the Orange column compared to the Blue column. -> Conclusion: Older adults tend to have higher levels of dissatisfaction.


**Explanation:**

- **hue='Churn'**: This is the most important parameter. It told Seaborn: "For each gender, separate into 2 color columns based on the Churn value (Yes/No)."



**Step 4: Service & Contract Analysis:**

This is our most important step, because it is usually where the root cause of customer churn lies.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.countplot(data=df, x='InternetService', hue='Churn', ax=axes[0], palette="Set2")
axes[0].set_title('Internet Services')

sns.countplot(data=df, x='Contract', hue='Churn', ax=axes[1], palette="Set2")
axes[1].set_title('Contract')

sns.countplot(data=df, x='PaymentMethod', hue='Churn', ax=axes[2], palette="Set2")
axes[2].set_title('Payment Method')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

**Chart analysis:**

- **Internet Services**: The fiber optic group has a sudden spike in tall orange columns. This is very strange because fiber optic is usually better than DSL. Perhaps the price is too high or the quality is unstable?

- **Contract**: The Month-to-month group has an extremely high churn rate. Conversely, the 2-year commitment group was almost completely loyal.

- **Payment Method**: The group using electronic checks dropped out the most.

**Explanation:**

- **tick_params(rotation=45)**: The payment method labels are quite long (e.g., "Bank transfer (automatic)"), and rotating them prevents them from overlapping.

**Step 5: Analyze the Churn Rate by breaking down all Services:**

In [ ]:
services = [
    'PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup',
    'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies'
]

colors = ['#FFF5BA', '#D7BDE2']
labels = ['No', 'Yes']

plt.figure(figsize=(12, 7))

for service in services:
    counts = df[service].value_counts()
    percents = counts / counts.sum() * 100

    unique_n = len(counts)
    service_colors = plt.cm.Pastel1(range(unique_n))

    bottom = 0
    for idx, category in enumerate(counts.index):
        val = counts[category]
        pct = percents[category]

        plt.bar(service, val, bottom=bottom, color=service_colors[idx])
        plt.text(service, bottom + val/2, f"{val}\n({pct:.1f}%)",
                 ha='center', va='center', fontsize=10)
        bottom += val

plt.ylabel('Count')
plt.title('Customer Churn Analysis by Services')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.xticks(rotation=45)
plt.legend(labels=labels, title='Churn')
plt.tight_layout()
plt.show()

**Chart Analysis:**

**1. PhoneService:**

- This column is almost dominated by one color (Red) (about 90%)
- Meaning: This is a telecommunications company, almost everyone has a phone service.

**2. Security & Support Group (OnlineSecurity, OnlineBackup, DeviceProtection, TechSupport):**

- The color corresponding to "No" (Not registered) will be the majority (usually > 60-70%).

- Meaning: Customers tend to only buy basic network packages and are less interested in technical/security value-added services.

**3. Entertainment Group (StreamingTV, StreamingMovies):**

- The Yes/No ratio is usually around 40/60.

- Meaning: Entertainment needs may be higher than security needs.

**Step 6: Analyze Numerical Data (Money & Time):**

Consider the distribution of fares and length of service.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sns.kdeplot(data=df, x='tenure', hue='Churn', fill=True, ax=axes[0], palette="coolwarm")
axes[0].set_title('tenure Distribution')

sns.boxplot(data=df, x='Churn', y='MonthlyCharges', hue='Churn', ax=axes[1], palette="coolwarm", legend=False)
axes[1].set_title('Monthly Charges')

plt.show()

**Chart analysis:**

**Tenure (KDE Plot)**: You'll see the red peak (Churn) spike up at tenure = 0-5 (months). Meaning customers often abandon the product right after trying it.

**Monthly Charges (Boxplot)**: The red box (Yes) is located higher than the blue box (No). On average, those who leave have to pay higher monthly fees than those who stay.

**Explanation:**

**sns.kdeplot (Kernel Density Estimate)**: Plots the probability density curve. It helps to see the "shape" of the data more smoothly than a histogram.

**sns.boxplot**: Displays the interquartile range (25% - 75%). It helps compare the median (the horizontal line in the box) between two groups very quickly.

**Step 7: Analyze the Relationship between Tenure and Monthly Charges according to Churn:**

In [ ]:
plt.figure(figsize=(10,6))
sns.scatterplot(data=df, x='tenure', y='MonthlyCharges', hue='Churn', alpha=0.6)
plt.title("The relationship between Tenure and Monthly Charges according to Churn")
plt.xlabel("tenure")
plt.ylabel("Monthly Charges")
plt.legend(title="Churn")
plt.show()

**Chart Analysis:**

- The scatter plot shows that churned customers (marked in orange) tend to cluster most heavily where tenure is low (new customers) and MonthlyCharges are high. This suggests that new customers who face higher monthly fees are more likely to leave early.

- Conversely, customers with longer tenure (those who have stayed for a long time) remain mostly stable across all MonthlyCharges levels. This indicates that once customers stay long enough, pricing becomes less influential on their decision to churn.

**Explantion:**

**sns.scatterplot**: Plots individual data points on an x–y plane. It is useful for showing relationships between two numerical variables, and the hue parameter highlights category differences (e.g., churn vs. non-churn).

**Step 8: Use Correlation Matrix to find the Correlation between variables:**

In [ ]:
df_encoded = df.drop(columns=['customerID'], errors='ignore').copy()

le = LabelEncoder()
df_encoded[df_encoded.select_dtypes('object').columns] = \
    df_encoded.select_dtypes('object').apply(le.fit_transform)

plt.figure(figsize=(14,10))
sns.heatmap(df_encoded.corr(numeric_only=True), annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Heatmap: Correlation between variables')
plt.show()

**Chart Analysis**

- **Colors and Strength:** The color intensity and shade indicate the strength and direction of the relationship. A value close to +1 signifies a strong positive correlation, -1 a strong negative correlation, and 0 indicates no linear correlation.

- **Key Relationships:** We are particularly interested in the Churn row (or column) to see which features have the strongest positive or negative correlation with customer churn.

## **Looking back #2:**

Through various visualizations including pie charts, bar charts, KDE plots, box plots, and scatter plots, we have gained significant insights into customer churn behavior. This exploratory data analysis has revealed several patterns and relationships between customer attributes, service usage, financial aspects, and their likelihood to churn.

From these patterns, we can distinctly identify two key customer profiles:

**Customer Profiles:**

**1. 'High Risk Profile' Customer Persona:** This group exhibits the highest potential for churn and is a primary target for retention efforts. They are typically:

- New customers (Newbies), using the service for less than 12 months, with a particularly high churn rate in the first 0-6 months.
- Paying very high monthly fees (e.g., > $70/month).
- Using fiber optic internet service (suggesting potential issues with this service offering).
- On a month-to-month contract and paying with an electronic check.
- Often senior citizens or single individuals (without a partner or dependents).

**2. 'Loyal' Customer Profile:** This group forms the stable foundation of the customer base, providing consistent revenue. They are typically:

- Long-term customers with tenure greater than 2 years.
- Paying low or medium monthly fees (e.g., < $60/month).
- Using DSL network or no internet service.
- On 1-2 year contracts with automatic payment methods (Bank transfer/Credit card).

**Next Steps:**
Given these insights and the class imbalance identified (only about 27% of customers churn), our immediate next steps will focus on Section 3: Data Preprocessing, followed by Machine Learning Modeling. Our primary goal will be to develop models that effectively identify the 'High Risk Profile' customers. This means we will prioritize optimizing Recall for the Churn group to minimize false negatives (missing customers who are about to leave), rather than just focusing on overall accuracy. We plan to explore various models like Logistic Regression, Random Forest, XGBoost, LightGBM, and CatBoost to achieve this objective.

##**Section 3. Data Preprocessing:**

**Step 1: Data cleaning: Clean TotalCharges and Remove customerID**

In [ ]:
df.describe(include='all').T

In [ ]:
df[(df['InternetService'] == 'No') & (
    (df['OnlineSecurity'] == 'Yes') |
    (df['OnlineBackup'] == 'Yes') |
    (df['DeviceProtection'] == 'Yes') |
    (df['TechSupport'] == 'Yes') |
    (df['StreamingTV'] == 'Yes') |
    (df['StreamingMovies'] == 'Yes')
)]

In [ ]:
df[(df['PhoneService'] == 'No') & (df['MultipleLines'] == 'Yes')]

In [ ]:
df[(df['tenure'] == 0) & (pd.to_numeric(df['TotalCharges'], errors='coerce') == 0)]

In [ ]:
df['calc_total'] = df['MonthlyCharges'] * df['tenure']
df[['TotalCharges', 'calc_total']].head(20)

In [ ]:
df = df.drop(columns=['customerID'], errors='ignore')

In [ ]:
blank_mask = df["TotalCharges"].astype(str).str.strip() == ""
num_blank = blank_mask.sum()
print("Number of blank TotalCharges lines:", num_blank)

blank_tenure_check = df.loc[blank_mask, "tenure"].unique()
print("tenure in blank lines:", blank_tenure_check)

df.loc[blank_mask, "TotalCharges"] = np.nan
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

df["expected_total"] = df["MonthlyCharges"] * df["tenure"]

print("expected_total in missing lines:", df.loc[df["TotalCharges"].isna(), "expected_total"].unique())

non_missing = df.dropna(subset=["TotalCharges"])
non_missing["diff"] = non_missing["TotalCharges"] - non_missing["expected_total"]

print("\nDiff statistics:")
print(non_missing["diff"].describe())

corr_mc = non_missing["diff"].corr(non_missing["MonthlyCharges"])
corr_ten = non_missing["diff"].corr(non_missing["tenure"])
print("\nDiff correlation coefficient with MonthlyCharges:", corr_mc)
print("Diff correlation coefficient with tenure:", corr_ten)

non_missing["relative_diff"] = non_missing["diff"] / non_missing["expected_total"].replace(0, np.nan)
print("\nCorrelation statistics (%):")
print(non_missing["relative_diff"].describe())

threshold = 5  # For example, a deviation of more than 5 USD is considered large.
num_big_diff = (non_missing["diff"].abs() > threshold).sum()
print(f"\nThe number of lines is more offset than {threshold} USD:", num_big_diff)


This code cell confirmed the initial hypotheses: the 11 empty 'TotalCharges' values ​​belonged to new customers (tenure = 0) and were correctly assigned the value 0. The small discrepancies in the other records simply reflected normal payment fluctuations and not a data issue.

In [ ]:
df.loc[df["TotalCharges"].isna(), "TotalCharges"] = (
    df["MonthlyCharges"] * df["tenure"]
)

In [ ]:
df["TotalCharges"].isna().sum()

In [ ]:
print(df.head().to_markdown(index=False, numalign="left", stralign="left"))

The data cleaning steps in 'Section 3: Data Preprocessing' primarily focus on refining the dataset after the initial diagnosis. Here's a breakdown of the key cleaning actions:

**1. Handling 'TotalCharges' - The Core Cleaning Task:**

- Detecting and Converting Blank Strings to NaN: As identified in the diagnostic phase (Section 1, Step 5), the TotalCharges column had 11 entries that were empty strings (' ') rather than proper numeric values or standard NaNs. The first part of this cleaning process explicitly finds these blank strings and replaces them with np.nan.
- Coercing to Numeric: The entire TotalCharges column is then converted to a numeric type (float). The errors='coerce' argument ensures that any value that still cannot be converted (which shouldn't happen after handling blank strings, but acts as a safeguard) is also turned into np.nan.
- Imputing Missing 'TotalCharges': For the 11 rows where TotalCharges was np.nan (because they were originally blank and corresponded to customers with tenure = 0), these NaN values are filled by calculating MonthlyCharges * tenure. Since tenure was 0 for these customers, their TotalCharges effectively become 0.

**2. Removing 'customerID' (Cell YEGHB6JJq7dv):**

- The customerID column, which is a unique identifier for each customer, is dropped from the DataFrame. This is a standard preprocessing step as customer IDs are typically not useful features for machine learning models and can sometimes even introduce noise or unintended patterns if not handled correctly.

These steps ensure that the TotalCharges column is in a consistent, numeric format suitable for calculations and modeling, and that irrelevant identification columns are removed, preparing the data for further feature engineering and model training.

**Step 2: Categorize numerical and categorical features:**

In [ ]:
target = 'Churn'

X = df.drop(columns=['Churn'])
y = df['Churn']

y = y.map({'Yes': 1, 'No': 0})

numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()

if 'TotalCharges' in categorical_features:
    categorical_features.remove('TotalCharges')
    numeric_features.append('TotalCharges')

if target in categorical_features:
    categorical_features.remove(target)

if 'SeniorCitizen' in numeric_features:
    numeric_features.remove('SeniorCitizen')
    categorical_features.append('SeniorCitizen')

print("=== Numeric features ===")
print(numeric_features)

print("\n=== Categorical features ===")
print(categorical_features)

Step 2 in the Data Preprocessing section is all about categorizing numerical and categorical features and setting up the target variable:

1. Target Variable Identification: The Churn column is explicitly defined as the target variable (y).
2. Feature-Target Split: The dataset is divided into two main parts: X (containing all features except Churn) and y (containing only the Churn target variable).
3. Target Encoding: The Churn target variable is converted from its original 'Yes'/'No' string values into numerical 1s and 0s, respectively. This is necessary for machine learning models that require numerical inputs.
4. Automatic Feature Type Classification: The code automatically identifies and separates all features in X into two lists:
- numeric_features: Columns detected as int64 or float64 data types.
- categorical_features: Columns detected as object, category, or bool data types.
5. Data Type Consistency Check: A crucial safeguard is included to ensure that TotalCharges (which was previously cleaned to be numeric) is correctly classified as a numeric feature, even if it might have initially been misidentified as categorical.

This step is essential because different types of features require different preprocessing techniques (e.g., scaling for numerical, one-hot encoding for categorical), which will be handled in the subsequent pipeline definition.



**Step 3: Define Pipelines:**

In [ ]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

print("\n=== Numeric transformer ===")
display(numeric_transformer)

In [ ]:
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

print("\n=== Categorical transformer ===")
display(categorical_transformer)

In [ ]:
preprocess = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

print("\n=== Preprocess ColumnTransformer ===")
display(preprocess)

Step 3 in the Data Preprocessing section focuses on defining the preprocessing pipelines for both numerical and categorical features. This is a critical step that sets up a systematic way to clean and transform our data consistently.

**1. Numeric Transformer Pipeline:** This pipeline is designed for all numerical features. It consists of two main steps:

- SimpleImputer(strategy='median'): This step handles any potential missing numerical values by replacing them with the median of their respective columns. The median is often used to be robust against outliers.
- StandardScaler(): This step scales the numerical features so they have a mean of 0 and a standard deviation of 1. Scaling is important for many machine learning algorithms as it prevents features with larger values from dominating the learning process.

**2. Categorical Transformer Pipeline:** This pipeline is designed for all categorical features. It also has two main steps:

- SimpleImputer(strategy='most_frequent'): This step handles any potential missing categorical values by replacing them with the most frequently occurring category in their respective columns.
- OneHotEncoder(handle_unknown='ignore'): This step converts categorical features into a numerical format that machine learning models can understand. Each unique category is transformed into a new binary column, with 1 indicating the presence of that category and 0 otherwise. handle_unknown='ignore' ensures that if a new, unseen category appears in the test set, it won't cause an error.

**3. ColumnTransformer (preprocess):** This is the overarching component that brings the two individual pipelines together. It applies the numeric_transformer to all numeric_features and the categorical_transformer to all categorical_features. This ensures that each type of feature receives the appropriate preprocessing steps, effectively transforming the raw input data into a clean, scaled, and encoded format suitable for machine learning algorithms.



**Step 4: Separate training and testing sets:**

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f"X_train: {X_train.shape}")
print(f"X_test: {X_test.shape}")

Step 4 in Section 3, 'Data Preprocessing', is about separating the training and testing sets.

This crucial step involves dividing the cleaned and preprocessed dataset (X and y) into two distinct subsets:

1. Training Set (X_train, y_train): This portion of the data (80% in this case) is used to train the machine learning models. The models learn patterns and relationships from this data.
2. Testing Set (X_test, y_test): This portion of the data (20% in this case) is held back and used only to evaluate the performance of the trained models. It simulates unseen data, allowing us to assess how well the model generalizes to new, real-world examples.

The train_test_split function from sklearn.model_selection is used for this purpose, with random_state=42 for reproducibility and stratify=y to ensure that the proportion of churned customers (Yes/No) is maintained similarly in both the training and testing sets, which is particularly important for imbalanced datasets.

## **Looking back #3:**

**We successfully completed the data preprocessing steps, which involved:**

- Cleaning the TotalCharges column: We addressed the 11 non-numeric values by setting them to 0, particularly for new customers with tenure = 0, ensuring this column is now numeric.
- Removing customerID: This unique identifier is not useful for modeling and was dropped.
- Categorizing Features: We clearly separated numerical and categorical features for appropriate handling.
- Defining Pipelines: We set up robust preprocessing pipelines for both numerical features (imputation with median, scaling) and categorical features (imputation with most frequent, one-hot encoding).
- Splitting Data: We split the data into training and testing sets (X_train, X_test, y_train, y_test) to ensure proper model evaluation.
- Target Variable Transformation: The Churn target variable was successfully converted to numerical 0 (No) and 1 (Yes).

**Key Outcome:**

After preprocessing, we have effectively transformed our raw data into a clean, structured format ready for machine learning. The dataset now consists of 23 features, including numerical and one-hot encoded categorical variables. This size (23 features, 7000 rows) is suitable for various standard machine learning algorithms.

**Next Steps:**

Our next steps will focus on Machine Learning Modeling, specifically addressing the challenge of class imbalance (only about 27% churn). Our strategy is to:

1. Prioritize Recall for the Churn group: This means we aim to correctly identify as many actual churners as possible, even if it leads to some false positives, as missing a potential churner is more costly.
2. Employ a multi-model approach: We will start with Logistic Regression as a baseline for interpretability, then explore more complex models like Random Forest and XGBoost to capture non-linear relationships and potentially improve performance.

The goal is to build models that can effectively identify 'High Risk Profile' customers, enabling proactive retention efforts.

##**Section 4: Machine Learning Modeling:**

### **1: Using the "RandomForestClassifier" Machine Learning model:**

In [ ]:
target = 'Churn'
X = df.drop(columns=[target])
y = df[target]

if 'TotalCharges' in categorical_features:
    categorical_features.remove('TotalCharges')
    numeric_features.append('TotalCharges')

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocess = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

rf_params = {
        'n_estimators': 400,
        'max_depth': 12,
        'min_samples_split': 10,
        'min_samples_leaf': 4,
        'max_features': 'sqrt',
        'class_weight': 'balanced',
        'bootstrap': True,
        'n_jobs': -1,
        'random_state': 42
    }

pipeline = ImbPipeline(steps=[
        ('preprocess', preprocess),
        ('classifier', RandomForestClassifier(**rf_params))
    ])
pipeline

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

print(">>> Training Random Forest (Class Weight Balanced)...")
pipeline.fit(X_train, y_train)
print(">>> Training Complete.")

rf_y_pred = pipeline.predict(X_test)
rf_y_proba = pipeline.predict_proba(X_test)[:, 1]

print("\n=== CLASSIFICATION REPORT (Original Model) ===")
print(classification_report(y_test, rf_y_pred))

print('=== CONFUSION MATRIX (Original Model) ===')
cm = confusion_matrix(y_test, rf_y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['No Churn', 'Churn'],
            yticklabels=['No Churn', 'Churn'])
ax.set_title('RandomForest Confusion Matrix')
ax.set_xlabel('Predicted Label')
ax.set_ylabel('True Label')
plt.tight_layout()
plt.show()

auc_score = roc_auc_score(y_test, rf_y_proba)
print(f"\n=== ROC AUC SCORE (Original Model): {auc_score:.4f} ===")

**Classification Report:**

**Accuracy: The model achieved an overall accuracy of 74%.**

For customers who did not churn (class 0):

- Precision: 91% (Of all customers predicted not to churn, 91% actually didn't).

- Recall: 72% (Of all customers who didn't churn, 72% were correctly identified).

For customers who did churn (class 1):

- Precision: 51% (Of all customers predicted to churn, 51% actually did).

- Recall: 80% (Of all customers who actually churned, 80% were correctly identified). This is a good recall for the minority class, indicating the model is reasonably effective at capturing churners.

**Confusion Matrix:**

- True Negatives (TN): 747 customers were correctly predicted not to churn.

- False Positives (FP): 288 customers were incorrectly predicted to churn (Type I error).

- False Negatives (FN): 75 customers were incorrectly predicted not to churn (Type II error). This is important for churn prediction, as these are the customers we missed.

- True Positives (TP): 299 customers were correctly predicted to churn.
ROC AUC Score:

The model achieved an ROC AUC score of 0.8312, which indicates a good ability to distinguish between churn and non-churn classes.
In summary, the model shows good potential for identifying churners, with a respectable recall for the churn class and a strong ROC AUC score. This suggests it's a solid baseline for predicting customer churn.

### **2. Using the "XGBoost" Machine Learning model:**

There are two key actions that we need to achieve:

One, to calculate the scale_pos_weight.

Two, to define the XGBoost parameters.

These are crucial for handling class imbalance and configuring the model's behavior.

In [ ]:
neg_count = y_train.value_counts()[0]
pos_count = y_train.value_counts()[1]
spw = neg_count / pos_count
print(f">>> Calculated scale_pos_weight: {spw:.4f}")

In [ ]:
xgb_params = {
		'n_estimators': 100,
		'learning_rate': 0.05,
		'max_depth': 3,
		'min_child_weight': 2,
		'gamma': 0.2,
		'subsample': 0.8,
		'colsample_bytree': 0.8,
		'scale_pos_weight': spw,
		'eval_metric': 'logloss',
		'use_label_encoder': False,
		'random_state': 42,
		'n_jobs': -1
	}

In [ ]:
from xgboost import XGBClassifier
pipeline = ImbPipeline(steps=[
		('preprocess', preprocess),
		('classifier', XGBClassifier(**xgb_params))
	])
pipeline

In [ ]:
print(">>> Training XGBoost Model...")
pipeline.fit(X_train, y_train)
print(">>> Training Completed.")
xgb_y_pred = pipeline.predict(X_test)
xgb_y_proba = pipeline.predict_proba(X_test)[:, 1]

print("\n=== CLASSIFICATION REPORT ===")
print(classification_report(y_test, xgb_y_pred))

print("=== CONFUSION MATRIX ===")
plt.figure(figsize=(6, 5))
cm = confusion_matrix(y_test, xgb_y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
	            xticklabels=['No Churn', 'Churn'],
	            yticklabels=['No Churn', 'Churn'])
plt.title('XGBoost Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

auc = roc_auc_score(y_test, xgb_y_proba)
print(f"\n=== ROC AUC SCORE: {auc:.4f} ===")

**Classification Report:**

**Accuracy: The model achieved an overall accuracy of 75%.**

For customers who did not churn (class 0):

- Precision: 91% (Of all customers predicted not to churn, 91% actually didn't).

- Recall: 72% (Of all customers who didn't churn, 72% were correctly identified).

For customers who did churn (class 1):

- Precision: 52% (Of all customers predicted to churn, 52% actually did).

- Recall: 81% (Of all customers who actually churned, 81% were correctly identified). This is a strong recall for the minority class, indicating the model is highly effective at capturing churners.

**Confusion Matrix:**

- True Negatives (TN): 749 customers were correctly predicted not to churn.

- False Positives (FP): 286 customers were incorrectly predicted to churn (Type I error).

- False Negatives (FN): 70 customers were incorrectly predicted not to churn (Type II error). This is important for churn prediction, as these are the customers we missed.

- True Positives (TP): 304 customers were correctly predicted to churn.

The model achieved an ROC AUC score of 0.8461, which indicates a very good ability to distinguish between churn and non-churn classes. This model shows excellent potential for identifying churners, with an even higher recall for the churn class (81%) compared to the RandomForest model and a slightly better ROC AUC score (0.8461). This suggests it's a robust model for predicting customer churn, especially valuing the identification of customers likely to leave.

###**3. Using the "LightGBM" Machine Learning model:**

In [ ]:
n_pos = np.sum(y_train)
n_neg = len(y_train) - n_pos
spw1 = n_neg / n_pos
print('Calculated SPW is:', spw1)

In [ ]:
lgbm_params = {
    'n_estimators': 1000,
    'learning_rate': 0.05,
    'num_leaves': 30,
    'max_depth': 3,
    'min_child_samples': 20,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'scale_pos_weight': spw1,
    'random_state': 42,
    'n_jobs': -1,
    'verbosity': -1
    }

In [ ]:
from lightgbm import LGBMClassifier
pipeline = ImbPipeline(steps=[
        ('preprocess', preprocess),
        ('classifier', LGBMClassifier(**lgbm_params))
    ])
pipeline

In [ ]:
print(">>> Training LightGBM Model...")
pipeline.fit(X_train, y_train)
print(">>> Training Completed.")

In [ ]:
lgbm_y_pred = pipeline.predict(X_test)
lgbm_y_proba = pipeline.predict_proba(X_test)[:, 1]

print("\n=== CLASSIFICATION REPORT ===")
print(classification_report(y_test, lgbm_y_pred))

print("\n=== CONFUSION MATRIX ===")
cm = confusion_matrix(y_test, lgbm_y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
	            xticklabels=['No Churn', 'Churn'],
	            yticklabels=['No Churn', 'Churn'])
plt.title('LightGBM Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

auc = roc_auc_score(y_test, lgbm_y_proba)
print(f"\n=== ROC AUC SCORE: {auc:.4f} ===")

**Classification Report:**

**Accuracy: The model achieved an overall accuracy of 75%.**

For customers who did not churn (class 0):

- Precision: 90% (Of all customers predicted not to churn, 90% actually didn't).

- Recall: 75% (Of all customers who didn't churn, 75% were correctly identified).

For customers who did churn (class 1):

- Precision: 52% (Of all customers predicted to churn, 52% actually did).

- Recall: 76% (Of all customers who actually churned, 76% were correctly identified). This is a strong recall for the minority class, indicating the model is highly effective at capturing churners.

**Confusion Matrix:**

- True Negatives (TN): 773 customers were correctly predicted not to churn.

- False Positives (FP): 262 customers were incorrectly predicted to churn (Type I error).

- False Negatives (FN): 88 customers were incorrectly predicted not to churn (Type II error). This is important for churn prediction, as these are the customers we missed.

- True Positives (TP): 286 customers were correctly predicted to churn.

The model achieved an ROC AUC score of 0.8325, and demonstrates solid performance in identifying churners, achieving a churn recall of 76%. This indicates a good ability to capture a significant portion of customers likely to leave, which is valuable for retention efforts. With an overall accuracy of 75% and a churn precision of 52%, LightGBM provides a balanced approach to churn prediction, effectively identifying at-risk customers while maintaining a reasonable level of false positives. Its ROC AUC score of 0.8325 further confirms its good discriminative power."

###**4. Using the "Logistic Regression" Machine Learning model:**

In [ ]:


pipeline.set_params(
    classifier=LogisticRegression(
        solver='liblinear',
        penalty='l1',
        C=0.01,
        class_weight='balanced',
        max_iter=1000,
        random_state=42
    )
)

In [ ]:
print(">>> Training Logistic Regression Model...")
pipeline.fit(X_train, y_train)
print(">>> Training Completed.")

lr_y_pred = pipeline.predict(X_test)
lr_y_proba = pipeline.predict_proba(X_test)[:, 1]

print("\n=== CLASSIFICATION REPORT ===")
print(classification_report(y_test, lr_y_pred))

print("\n=== CONFUSION MATRIX ===")
cm = confusion_matrix(y_test, lr_y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
	            xticklabels=['No Churn', 'Churn'],
	            yticklabels=['No Churn', 'Churn'])
plt.title('Logistic Regression Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

auc = roc_auc_score(y_test, lr_y_proba)
print(f"\n=== ROC AUC SCORE: {auc:.4f} ===")

**Classification Report:**

**Accuracy: The model achieved an overall accuracy of 73%.**

For customers who did not churn (class 0):

- Precision: 91% (Of all customers predicted not to churn, 91% actually didn't).

- Recall: 70% (Of all customers who didn't churn, 70% were correctly identified).

For customers who did churn (class 1):

- Precision: 50% (Of all customers predicted to churn, 50% actually did).

- Recall: 82% (Of all customers who actually churned, 82% were correctly identified). This is a strong recall for the minority class, indicating the model is highly effective at capturing churners.

**Confusion Matrix:**

- True Negatives (TN): 724 customers were correctly predicted not to churn.

- False Positives (FP): 311 customers were incorrectly predicted to churn (Type I error).

- False Negatives (FN): 69 customers were incorrectly predicted not to churn (Type II error). This is important for churn prediction, as these are the customers we missed.

- True Positives (TP): 305 customers were correctly predicted to churn.

The model achieved an ROC AUC score of 0.8368, which indicates a very good ability to distinguish between churn and non-churn classes. The Logistic Regression model demonstrates excellent potential for identifying churners, boasting a high recall of 82% for the churn class—outperforming models like RandomForest in this critical metric—and achieving a strong ROC AUC score of 0.8368, making it a robust choice for proactive churn prediction.

###**5. Using the "CatBoost" Machine Learning model:**

In [ ]:
cat_model = CatBoostClassifier(
    iterations=600,
    depth=5,
    learning_rate=0.05,
    loss_function='Logloss',
    eval_metric='AUC',
    verbose=0,
    random_seed=42,
    scale_pos_weight=spw1
)

In [ ]:
cat_model = CatBoostClassifier(
    iterations=600,
    depth=5,
    learning_rate=0.05,
    loss_function='Logloss',
    eval_metric='AUC',
    verbose=0,
    random_seed=42,
    scale_pos_weight=spw1
)

In [ ]:
cat_pipeline = Pipeline(steps=[
    ('preprocessor', preprocess),
    ('model', cat_model)
])

cat_pipeline.fit(X_train, y_train)

cat_y_pred = cat_pipeline.predict(X_test)
cat_y_proba = cat_pipeline.predict_proba(X_test)[:, 1]

print("\n=== CLASSIFICATION REPORT ===")
print(classification_report(y_test, cat_y_pred))

print("\n=== CONFUSION MATRIX ===")
cm = confusion_matrix(y_test, cat_y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
	            xticklabels=['No Churn', 'Churn'],
	            yticklabels=['No Churn', 'Churn'])
plt.title('Catboost Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

auc = roc_auc_score(y_test, cat_y_proba)
print(f"\n=== ROC AUC SCORE: {auc:.4f} ===")

**Classification Report:**

**Accuracy: The model achieved an overall accuracy of 75%.**

For customers who did not churn (class 0):

- Precision: 91% (Of all customers predicted not to churn, 91% actually didn't).

- Recall: 74% (Of all customers who didn't churn, 74% were correctly identified).

For customers who did churn (class 1):

- Precision: 52% (Of all customers predicted to churn, 52% actually did).

- Recall: 79% (Of all customers who actually churned, 79% were correctly identified). This is a strong recall for the minority class, indicating the model is highly effective at capturing churners.

**Confusion Matrix:**

- True Negatives (TN): 762 customers were correctly predicted not to churn.

- False Positives (FP): 273 customers were incorrectly predicted to churn (Type I error).

- False Negatives (FN): 77 customers were incorrectly predicted not to churn (Type II error). This is important for churn prediction, as these are the customers we missed.

- True Positives (TP): 297 customers were correctly predicted to churn.

The CatBoost model achieved an ROC AUC score of 0.8461, and demonstrates strong performance in identifying churners, with a notable recall of 79% for the churn class. This robust recall ensures that a significant portion of at-risk customers are correctly identified, making it a valuable tool for proactive retention efforts. While achieving an overall accuracy of 75%, its effectiveness in pinpointing potential churners, despite a moderate precision of 52% for the churn class, underscores its utility where minimizing missed churners is paramount.


##**Section 5: Model Comparison and Recommendation:**

To compare the models comprehensively, we will first consolidate all extracted metrics (accuracy, precision, recall, F1-score for churn, and ROC AUC score) for Random Forest, XGBoost, LightGBM, Logistic Regression, and CatBoost into a single pandas DataFrame. This structured format will facilitate easy visualization and comparison.



Based on the consolidated metrics and visualizations, here's a comparison of the models:

In [ ]:
from sklearn.metrics import classification_report, roc_auc_score
import pandas as pd

def get_metrics(y_true, y_pred, y_proba, model_name):
    """Calculates and returns key classification metrics."""
    report = classification_report(y_true, y_pred, output_dict=True)
    accuracy = report['accuracy']
    churn_precision = report['1']['precision']
    churn_recall = report['1']['recall']
    churn_f1_score = report['1']['f1-score']
    roc_auc = roc_auc_score(y_true, y_proba)

    return {
        'Model': model_name,
        'Accuracy': accuracy,
        'Churn Precision': churn_precision,
        'Churn Recall': churn_recall,
        'Churn F1-score': churn_f1_score,
        'ROC AUC Score': roc_auc
    }

# Dynamically extract metrics for each model
metrics_list = []

# Random Forest
metrics_list.append(get_metrics(y_test, rf_y_pred, rf_y_proba, 'Random Forest'))

# XGBoost
metrics_list.append(get_metrics(y_test, xgb_y_pred, xgb_y_proba, 'XGBoost'))

# LightGBM
metrics_list.append(get_metrics(y_test, lgbm_y_pred, lgbm_y_proba, 'LightGBM'))

# Logistic Regression
metrics_list.append(get_metrics(y_test, lr_y_pred, lr_y_proba, 'Logistic Regression'))

# CatBoost
metrics_list.append(get_metrics(y_test, cat_y_pred, cat_y_proba, 'CatBoost'))

# Consolidate into a DataFrame
metrics_df = pd.DataFrame(metrics_list)

print("== Summary of ressults of 5 models ==:\n")
print(metrics_df.to_markdown(index=False, numalign="left", stralign="left"))

**Key Observations:**

1.  **Churn Recall (Identifying Churners):** XGBoost and Logistic Regression stand out with the highest Churn Recall at 0.79. This means they are best at identifying customers who *will* churn, which is crucial for a churn prediction model to enable proactive retention efforts.
2.  **ROC AUC Score (Overall Discriminative Power):** XGBoost and Logistic Regression also share the highest ROC AUC Score of 0.845, indicating excellent ability to distinguish between churners and non-churners across various probability thresholds.
3.  **Accuracy:** Random Forest and LightGBM show slightly higher overall accuracy (0.77), but this metric can be misleading in imbalanced datasets like this one, as it might prioritize the majority class (non-churners).
4.  **Churn Precision:** All models have relatively lower Churn Precision, meaning they predict some non-churners as churners (false positives). However, for churn prediction, high recall is generally prioritized over high precision to ensure fewer actual churners are missed.

**Recommendation:**

Considering the business objective of identifying customers likely to churn to enable retention strategies, **XGBoost** and **Logistic Regression** are the best-performing models. They offer the highest Churn Recall and ROC AUC Score, making them most effective at capturing actual churners. While Logistic Regression is simpler and more interpretable, XGBoost generally offers more flexibility and potentially better performance on complex relationships. For this dataset, their performance is nearly identical across key metrics.

Therefore, **XGBoost** is recommended for its strong performance on both Churn Recall and ROC AUC, alongside its robust handling of imbalanced data and general effectiveness in competitive machine learning tasks. Logistic Regression could also be a strong candidate, especially if model interpretability is a primary concern.

**Reasoning**:
Now that the metrics are consolidated, a bar chart visualizing Accuracy, Churn Recall, and ROC AUC Score for all models is neccessary. This will provide a clear visual comparison of the models' performance across key metrics.



In [ ]:
print("### Visualizing Model Performance Metrics:")

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

sns.barplot(x='Model', y='Accuracy', data=metrics_df, ax=axes[0], palette='viridis')
axes[0].set_title('Model Accuracy')
axes[0].set_ylabel('Accuracy')
axes[0].set_ylim(0, 1)
axes[0].tick_params(axis='x', rotation=45)

sns.barplot(x='Model', y='Churn Recall', data=metrics_df, ax=axes[1], palette='plasma')
axes[1].set_title('Model Churn Recall')
axes[1].set_ylabel('Recall (Churn)')
axes[1].set_ylim(0, 1)
axes[1].tick_params(axis='x', rotation=45)

sns.barplot(x='Model', y='ROC AUC Score', data=metrics_df, ax=axes[2], palette='magma')
axes[2].set_title('Model ROC AUC Score')
axes[2].set_ylabel('ROC AUC Score')
axes[2].set_ylim(0, 1)
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## Summary:

### Q&A
*   **Which model is recommended as the best-performing model for churn prediction?**
    XGBoost is recommended as the best-performing model for churn prediction, primarily due to its high Churn Recall and ROC AUC Score. Logistic Regression is also highlighted as a strong alternative with similar performance.

### Data Analysis Key Findings
*   XGBoost and Logistic Regression models exhibited the highest Churn Recall at 0.79, indicating superior ability to identify actual churning customers.
*   Both XGBoost and Logistic Regression also achieved the highest ROC AUC Score of 0.845, demonstrating excellent overall discriminative power between churners and non-churners.
*   Random Forest and LightGBM models showed slightly higher overall accuracy at 0.77, but this metric can be less reliable in imbalanced datasets.
*   All models displayed relatively lower Churn Precision, suggesting a tendency to generate false positives (predicting non-churners as churners), though for churn prediction, high recall is generally prioritized.


Umsampling positive class


In [ ]:
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.linear_model import LogisticRegression

# Assuming 'preprocess' is defined in a preceding cell and is globally available.
# It combines numeric_transformer and categorical_transformer.

pipeline = ImbPipeline(steps=[
    ('preprocess', preprocess),

    ('undersampler', RandomUnderSampler(sampling_strategy=0.9, random_state=42)),
    ('classifier', LogisticRegression(
        solver='liblinear',
        penalty='l1',
        C=0.01,
        max_iter=1000,
        random_state=42
    ))
])
pipeline

In [ ]:
print(">>> Training Logistic Regression Model...")
pipeline.fit(X_train, y_train)
print(">>> Training Completed.")

lr_y_pred = pipeline.predict(X_test)
lr_y_proba = pipeline.predict_proba(X_test)[:, 1]

print("\n=== CLASSIFICATION REPORT ===")
print(classification_report(y_test, lr_y_pred))

print("\n=== CONFUSION MATRIX ===")
cm = confusion_matrix(y_test, lr_y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
	            xticklabels=['No Churn', 'Churn'],
	            yticklabels=['No Churn', 'Churn'])
plt.title('Logistic Regression Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

auc = roc_auc_score(y_test, lr_y_proba)
print(f"\n=== ROC AUC SCORE: {auc:.4f} ===")

Đặt rule-based


In [ ]:
n_pos = np.sum(y_train)
n_neg = len(y_train) - n_pos
spw1 = n_neg / n_pos
print('Calculated SPW is:', spw1)

In [ ]:
xgb_params = {
		'n_estimators': 100,
		'learning_rate': 0.05,
		'max_depth': 3,
		'min_child_weight': 2,
		'gamma': 0.2,
		'subsample': 0.8,
		'colsample_bytree': 0.8,
		'scale_pos_weight': spw,
		'eval_metric': 'logloss',
		'use_label_encoder': False,
		'random_state': 42,
		'n_jobs': -1
	}

In [ ]:
from xgboost import XGBClassifier
pipeline = ImbPipeline(steps=[
		('preprocess', preprocess),
		('classifier', XGBClassifier(**xgb_params))
	])
pipeline

In [ ]:
rule_mask = (
    (X_test['Contract'] == 'Month-to-month') &
    (X_test['tenure'] < 6)
)

# Initialize y_pred_hybrid as a mutable copy of the XGBoost predictions
y_pred_hybrid = xgb_y_pred.copy()

# Apply the rule-based override: set prediction to 1 (churn) for matching customers
y_pred_hybrid[rule_mask] = 1


In [ ]:
print(">>> Training XGBoost Model...")
pipeline.fit(X_train, y_train)
print(">>> Training Completed.")
xgb_y_pred = pipeline.predict(X_test)
xgb_y_proba = pipeline.predict_proba(X_test)[:, 1]

print("\n=== CLASSIFICATION REPORT ===")
print(classification_report(y_test, xgb_y_pred))

print("=== CONFUSION MATRIX ===")
plt.figure(figsize=(6, 5))
cm = confusion_matrix(y_test, xgb_y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
	            xticklabels=['No Churn', 'Churn'],
	            yticklabels=['No Churn', 'Churn'])
plt.title('XGBoost Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

auc = roc_auc_score(y_test, xgb_y_proba)
print(f"\n=== ROC AUC SCORE: {auc:.4f} ===")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

print("=== ML ONLY ===")
print(classification_report(y_test, xgb_y_pred)) # Changed y_pred_ml to xgb_y_pred
print(confusion_matrix(y_test, xgb_y_pred)) # Changed y_pred_ml to xgb_y_pred

print("\n=== HYBRID OVERRIDE ===")
print(classification_report(y_test, y_pred_hybrid))
print(confusion_matrix(y_test, y_pred_hybrid))
